# The Workload You Do Not Know

*Choosing an architecture from an estimate.*

The previous topic ended with a lookup: fix the price and the deployment size, find your workload
mixture on the simplex, read off the architecture. It treated the mixture as **known**.

You do not know it. You estimate it from traffic, and a partition is precisely the structure that
punishes estimation error hardest — because the map from workload to architecture is piecewise
constant, error is *free* in a cell interior and flips the answer outright at a boundary.

Nothing is re-measured here. The module imports `rag_architecture_pareto` and layers uncertainty
over its decision layer.

```
uv run --with numpy --with scipy --with jupyter \
    jupyter execute notebooks/rag-architecture-workload-uncertainty/01_rag_architecture_workload_uncertainty.ipynb
```


In [ ]:
import pathlib
import sys

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / "rag_architecture_workload_uncertainty.py").exists()
                       else pathlib.Path.cwd() / "notebooks" / "rag-architecture-workload-uncertainty"))
import rag_architecture_workload_uncertainty as W

print(f"arms       : {', '.join(W.ARM_NAMES)}")
print(f"conditions : {', '.join(W.CONDITIONS)}")
print(f"prices pinned at the predecessor's headline: lambda={W.LAMBDA_HEADLINE:g}, rho={W.RHO_HEADLINE:g}")

## 1. Three rules, and one of them is not a rule

Plug-in commits to the estimate. Bayes averages over the posterior. Minimax hedges against the
worst regret it could come to feel.

That looks like three ideas. Watch what happens across the boundary the predecessor measured.

In [ ]:
print(f"{'bridge share':>13s} {'plug-in':>11s} {'Bayes':>11s} {'minimax':>11s}")
for f in (0.20, 0.34, 0.40, 0.43, 0.46, 0.52, 0.85):
    r = W.all_three(W.bridge_family(f), W.N_HEADLINE_OBS)
    mark = "" if r["plug_in"] == r["minimax"] else "   <- the hedge parts company"
    print(f"{f:13.2f} {r['plug_in']:>11s} {r['bayes']:>11s} {r['minimax']:>11s}{mark}")

## 2. Why Bayes is not a third rule

Utility is **affine** in the workload, so the expected utility of an arm over *any* posterior
equals its utility at the posterior **mean** — exactly, not approximately. The Bayes rule is
therefore the plug-in rule applied to a shrunk estimate, and the shrinkage is the whole of the
difference between them.

In [ ]:
W.test_bayes_is_plug_in_at_the_posterior_mean()
print("identity verified to < 1e-12 on every workload and every n tested")
print()
print("and the shrinkage is closed form -- it is the entire gap:")
for n in (10, 40, 160, 1280):
    print(f"  n={n:5d}  posterior-mean bridge share = {W.shrinkage(W.bridge_family(0.43), n):.4f}"
          f"   (the estimate was 0.4300)")

That identity is also what makes the simulation affordable. Evaluating two thousand posterior
draws by calling the imported `utility` once apiece took the module **two minutes**; writing the
same thing as the affine map it is takes **one matrix product**.

In [ ]:
W.test_kernel_matches_the_imported_utility()
print("the vectorized kernel reproduces the imported utility to < 1e-12, draw by draw")
M, const = W.utility_kernel()
print(f"  kernel M: {M.shape[0]} arms x {M.shape[1]} conditions, plus a per-arm constant")
print(f"  only one arm carries a constant at all (the offline index build): {np.argmin(const)} -> {const.min():.2e}")

## 3. The hedge is not a safe default

A minimax rule is often assumed to retreat to something bland. Measured, it tracks where the
posterior sits: deep inside a cell it *is* that cell's arm, and the cells hold different arms.

In [ ]:
for label, v in W.hedge_is_local().items():
    flag = "" if v["agree"] else "   <- diverges"
    print(f"  {label:>13s}  plug-in={v['plug_in']:>11s}  minimax={v['minimax']:>11s}{flag}")
print()
print(f"over workloads drawn without regard to where the boundaries are, the two rules")
print(f"differ on {W.disagreement_rate():.1%} of them")

## 4. How much traffic is enough — and where the question has no comfortable answer

In [ ]:
for label, w in (("deep inside a cell", W.bridge_family(0.85)),
                 ("uniform", W.uniform_within_regime()),
                 ("local-only", W.local_only()),
                 ("ON the boundary", W.bridge_family(0.43))):
    print(f"  {label:>19s}: rules agree from n = {W.agreement_n(w)}")
print()
print("worst-case regret as data accumulates, on the boundary:")
curve = W.regret_decay(W.bridge_family(0.43))
for n, r in curve:
    print(f"  n={n:5d}  {r:.4f}  {'#' * int(round(r * 200))}")
print()
print(f"halving ratios {W.halving_ratios(curve)}")
print(f"a clean 1/sqrt(n) law would hold these at {1/np.sqrt(2):.3f} -- they approach it from")
print("ABOVE rather than sitting on it, so the claim is the decay and the tail, not the rate.")

## 5. What the hedge costs

Both Bayes and minimax shrink toward the middle of the simplex. That is a **bet** that the traffic
is more central than the data says — and the bet can lose.

In [ ]:
print(f"{'bridge share':>13s} {'plug-in':>9s} {'Bayes':>9s} {'minimax':>9s}")
for r in W.cost_of_hedging():
    mark = "   <- hedging LOSES here" if r["hedge_worse"] else ""
    print(f"{r['frac']:13.2f} {r['plug_in']:9.5f} {r['bayes']:9.5f} {r['minimax']:9.5f}{mark}")
print()
print("Below the boundary the bet pays. Above it, where the truth sits in a narrow cell, the")
print("plug-in rule was correctly confident and the hedge drags it out -- and the loss there is")
print("LARGER than the gain below. A hedge here is a trade, not cheap insurance.")

## 6. Every claim, as an assertion

Including the collapse anchors that pin this topic to its predecessor: a point-mass posterior
makes all three rules agree and equal to the pareto topic's `winner()`.

In [ ]:
W._run_tests()